# IMPORT LIBRARY

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

# CONNECTING TO DATABASE

In [ ]:
load_dotenv("../.env")

conn = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connection created")

# COHORT RETENTION RATE(%)

##
RETRIVE DATA

In [ ]:
cust_activity = '''
        WITH date1 AS (
                SELECT customer_id
                        ,MIN(DATE_TRUNC('MONTH',event_ts)::DATE) AS date_min
                FROM interactions
                GROUP BY customer_id 
        )
        SELECT i.customer_id
                ,cm.month AS monthyear
        FROM customers_monthly_metrics cm
        LEFT JOIN date1 i ON i.customer_id = cm.customer_id
        WHERE cm.month<date_min

        UNION

        SELECT customer_id
                ,DATE_TRUNC('MONTH',event_ts)::DATE AS monthyear
        FROM interactions i
        WHERE DATE_PART('year',event_ts)<=2024

        ORDER BY customer_id,monthyear
    '''

df_cust_activity = pd.read_sql(cust_activity,conn)

##
CUSTOMER RETENTION ANALYSIS

In [ ]:
df_cust_activity['monthyear'] = pd.to_datetime(df_cust_activity['monthyear'])
df_cust_activity['cohort'] = df_cust_activity['monthyear'].dt.to_period('M')
df_cust_activity['first_cohort'] = df_cust_activity.groupby('customer_id')['cohort'].transform('min')
df_cust_activity['cohort_index'] = (df_cust_activity['cohort'] - df_cust_activity['first_cohort']).apply(lambda x: x.n)

In [ ]:
cohort_data = df_cust_activity.groupby(['first_cohort', 'cohort_index'])['customer_id'].nunique().reset_index()
cohort_counts = cohort_data.pivot(index='first_cohort', columns='cohort_index', values='customer_id')
retention_matrix = cohort_counts.divide(cohort_counts[0], axis=0)

In [ ]:
plt.figure(figsize=(22,14))

sns.heatmap(
    retention_matrix,
    annot=True,                  
    fmt=".0%",                   
    linewidths=0.5,
    linecolor='white',
    cmap="Blues",               
    mask=retention_matrix.isnull(),
    annot_kws={"size": 8},         
    cbar_kws={'label': 'Retention Rate'}  
)

plt.title("Cohort Analysis Heatmap", fontsize=16)
plt.ylabel("First Cohort")
plt.xlabel("Cohort Month")
plt.show()

##
CUSTOMER CHURN ANALYSIS

In [ ]:

retention_percentage = (retention_matrix*100).round(2)
churn_matrix = 100 - retention_percentage

display(churn_matrix)
plt.figure(figsize=(10, 6))

for cohort in churn_matrix.index[:6]: 
    plt.plot(churn_matrix.columns, 
             churn_matrix.loc[cohort], 
             marker='o', 
             label=str(cohort))

plt.title("Customer Inactivity Rate over Time")
plt.xlabel("Cohort Index (Bulan ke-)")
plt.ylabel("Inactivity Rate (%)")
plt.legend(title="First Cohort")
plt.grid(True)
plt.show()